# 01 · Data Overview & Data Quality

**Project:** Predicting term-deposit subscription (UCI Bank Marketing, `bank-full.csv`)

This notebook loads the raw data, documents every column, and checks data quality.
It does **not** change the data; all cleaning decisions are recorded for the preprocessing stage.

In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)

## 1. Load the data

In [2]:
df = pd.read_csv('../data/bank-full.csv')
print(df.shape)
df.head()

(45211, 17)


,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,58,management,married,tertiary,no,2143,yes,no,unknown,5,may,261,1,-1,0,unknown,no
1,44,technician,single,secondary,no,29,yes,no,unknown,5,may,151,1,-1,0,unknown,no
2,33,entrepreneur,married,secondary,no,2,yes,yes,unknown,5,may,76,1,-1,0,unknown,no
3,47,blue-collar,married,unknown,no,1506,yes,no,unknown,5,may,92,1,-1,0,unknown,no
4,33,unknown,single,unknown,no,1,no,no,unknown,5,may,198,1,-1,0,unknown,no


## 2. Data dictionary

Each row is one client at the time of a contact in the **current** phone campaign.

| Column | Type | Description |
| --- | --- | --- |
| age | numeric | Client age (years) |
| job | categorical | Type of job |
| marital | categorical | Marital status |
| education | categorical | Education level |
| default | binary | Has credit in default? |
| balance | numeric | Average yearly balance (euros) |
| housing | binary | Has a housing loan? |
| loan | binary | Has a personal loan? |
| contact | categorical | Contact channel (cellular / telephone / unknown) |
| day | numeric | Day of the month of the last contact |
| month | categorical | Month of the last contact |
| duration | numeric | Length of the last call in seconds (**known only after the call**) |
| campaign | numeric | Contacts in this campaign, including the last one |
| pdays | numeric | Days since last contact in a previous campaign (`-1` = never contacted) |
| previous | numeric | Contacts before this campaign |
| poutcome | categorical | Outcome of the previous campaign |
| **y** | binary | **Target:** did the client subscribe to a term deposit? |

## 3. Data types

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 45211 entries, 0 to 45210
Data columns (total 17 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   age        45211 non-null  int64
 1   job        45211 non-null  str  
 2   marital    45211 non-null  str  
 3   education  45211 non-null  str  
 4   default    45211 non-null  str  
 5   balance    45211 non-null  int64
 6   housing    45211 non-null  str  
 7   loan       45211 non-null  str  
 8   contact    45211 non-null  str  
 9   day        45211 non-null  int64
 10  month      45211 non-null  str  
 11  duration   45211 non-null  int64
 12  campaign   45211 non-null  int64
 13  pdays      45211 non-null  int64
 14  previous   45211 non-null  int64
 15  poutcome   45211 non-null  str  
 16  y          45211 non-null  str  
dtypes: int64(7), str(10)
memory usage: 5.9 MB


In [4]:
num_cols = df.select_dtypes(include='number').columns.tolist()
cat_cols = df.select_dtypes(exclude='number').columns.tolist()
print(f'{len(num_cols)} numeric columns:', num_cols)
print(f'{len(cat_cols)} categorical columns:', cat_cols)

7 numeric columns: ['age', 'balance', 'day', 'duration', 'campaign', 'pdays', 'previous']
10 categorical columns: ['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'poutcome', 'y']


## 4. Nulls and duplicates

In [5]:
print('Total null values:', df.isna().sum().sum())
print('Duplicate rows   :', df.duplicated().sum())

Total null values: 0
Duplicate rows   : 0


## 5. Hidden missing values

There are no nulls, but missing information is coded as the word `unknown` and as `pdays = -1`.

In [6]:
unknown = (df[cat_cols] == 'unknown').sum()
unknown = unknown[unknown > 0].to_frame('rows')
unknown['share_%'] = (unknown['rows'] / len(df) * 100).round(1)
unknown.sort_values('rows', ascending=False)

,rows,share_%
poutcome,36959,81.7
contact,13020,28.8
education,1857,4.1
job,288,0.6


In [7]:
never = (df['pdays'] == -1)
print('pdays == -1            :', never.sum(), f'({never.mean():.1%})')
print('previous == 0          :', (df['previous'] == 0).sum())
print('both at the same time  :', (never & (df['previous'] == 0)).sum())
print('poutcome unknown but previous > 0:', ((df['poutcome'] == 'unknown') & (df['previous'] > 0)).sum())

pdays == -1            : 36954 (81.7%)
previous == 0          : 36954
both at the same time  : 36954
poutcome unknown but previous > 0: 5


`pdays = -1` happens **exactly** when `previous = 0`: these are clients never contacted before.
For the same clients `poutcome` is `unknown`, so the three columns share one piece of information.

## 6. Summary statistics

In [8]:
df.describe().T.round(2)

,count,mean,std,min,25%,50%,75%,max
age,45211.0,40.94,10.62,18.0,33.0,39.0,48.0,95.0
balance,45211.0,1362.27,3044.77,-8019.0,72.0,448.0,1428.0,102127.0
day,45211.0,15.81,8.32,1.0,8.0,16.0,21.0,31.0
duration,45211.0,258.16,257.53,0.0,103.0,180.0,319.0,4918.0
campaign,45211.0,2.76,3.10,1.0,1.0,2.0,3.0,63.0
pdays,45211.0,40.20,100.13,-1.0,-1.0,-1.0,-1.0,871.0
previous,45211.0,0.58,2.30,0.0,0.0,0.0,0.0,275.0


In [9]:
df[cat_cols].describe().T

,count,unique,top,freq
job,45211,12,blue-collar,9732
marital,45211,3,married,27214
education,45211,4,secondary,23202
default,45211,2,no,44396
housing,45211,2,yes,25130
loan,45211,2,no,37967
contact,45211,3,cellular,29285
month,45211,12,may,13766
poutcome,45211,4,unknown,36959
y,45211,2,no,39922


## 7. Unique values per categorical column

In [10]:
for col in cat_cols:
    print(f'{col:10s}', df[col].nunique(), sorted(df[col].unique()))

job        12 ['admin.', 'blue-collar', 'entrepreneur', 'housemaid', 'management', 'retired', 'self-employed', 'services', 'student', 'technician', 'unemployed', 'unknown']
marital    3 ['divorced', 'married', 'single']
education  4 ['primary', 'secondary', 'tertiary', 'unknown']
default    2 ['no', 'yes']
housing    2 ['no', 'yes']
loan       2 ['no', 'yes']
contact    3 ['cellular', 'telephone', 'unknown']
month      12 ['apr', 'aug', 'dec', 'feb', 'jan', 'jul', 'jun', 'mar', 'may', 'nov', 'oct', 'sep']
poutcome   4 ['failure', 'other', 'success', 'unknown']
y          2 ['no', 'yes']


## 8. Findings

- **45,211 rows × 17 columns**: 7 numeric, 10 categorical. No nulls, no duplicate rows.
- Missing information is hidden as `unknown` (poutcome 81.7%, contact 28.8%, education 4.1%, job 0.6%) and as `pdays = -1` (81.7%).
- `pdays = -1`, `previous = 0` and `poutcome = unknown` all describe the same 36,954 never-contacted clients → one flag (`contacted_before`) captures it.
- `balance` has negative values (min -8,019); `duration` has 3 calls of 0 seconds.
- The target `y` is text (`yes` / `no`) and will be mapped to 1 / 0.